In [1]:
!pip install yfinance

In [21]:
import pandas as pd

merged = pd.read_parquet("/Users/andradaduta/Desktop/Master Thesis/Python Data/stoxx_600_returns_with_ff5_mom_daily_2010_2025.parquet")

display(merged.head())
print(merged.shape)

,gvkey,iid,date,isin,prccd,ajexdi,trfd,cshoc,curcdd,exchg,...,tri,ret,ret_w,Mkt-RF,SMB,HML,RMW,CMA,RF,Mom
0,001166,01W,2010-01-05,NL0000334118,18.025,1.0,1.004897,54275131.0,EUR,104,...,18.113271,0.008077,0.008077,-0.0007,0.0050,0.0094,-0.0036,-0.0009,0.0,0.0056
1,001166,01W,2010-01-06,NL0000334118,18.27,1.0,1.004897,54275131.0,EUR,104,...,18.359471,0.013501,0.013501,0.0047,0.0029,0.0053,-0.0009,-0.0015,0.0,0.0044
2,001166,01W,2010-01-07,NL0000334118,17.84,1.0,1.004897,54275131.0,EUR,104,...,17.927365,-0.023817,-0.023817,-0.0055,0.0042,0.0053,-0.0013,-0.0011,0.0,0.0040
3,001166,01W,2010-01-08,NL0000334118,18.485,1.0,1.004897,54275131.0,EUR,104,...,18.575524,0.035516,0.035516,0.0110,0.0021,0.0035,-0.0026,0.0016,0.0,0.0018
4,001166,01W,2010-01-11,NL0000334118,17.945,1.0,1.004897,54275131.0,EUR,104,...,18.03288,-0.029648,-0.029648,0.0070,0.0039,-0.0008,0.0033,-0.0012,0.0,-0.0018


(3763472, 35)


In [23]:
import pandas as pd
import yfinance as yf

# --------------------------------------------------
# 1. Currencies in your STOXX dataset
# --------------------------------------------------

currencies = ["EUR", "GBP", "SEK", "CHF", "DKK", "NOK", "PLN", "CZK", "USD", "ZAR"]

# --------------------------------------------------
# 2. Yahoo FX tickers
# Goal: fx_to_usd = 1 unit of local currency in USD
# --------------------------------------------------

fx_tickers = {
    "EUR": ("EURUSD=X", False),  # 1 EUR in USD
    "GBP": ("GBPUSD=X", False),  # 1 GBP in USD

    # These are quoted as 1 USD in local currency, so we invert
    "SEK": ("USDSEK=X", True),
    "CHF": ("USDCHF=X", True),
    "DKK": ("USDDKK=X", True),
    "NOK": ("USDNOK=X", True),
    "PLN": ("USDPLN=X", True),
    "CZK": ("USDCZK=X", True),
    "ZAR": ("USDZAR=X", True),
}

# --------------------------------------------------
# 3. Download period
# --------------------------------------------------

start_date = "2010-01-01"
end_date = "2025-12-31"

# --------------------------------------------------
# 4. Download FX rates
# --------------------------------------------------

fx_list = []

for cur in currencies:

    if cur == "USD":
        temp = merged[["date"]].drop_duplicates().copy()
        temp["curcdd"] = "USD"
        temp["fx_to_usd"] = 1.0
        fx_list.append(temp)
        print("USD: set fx_to_usd = 1")
        continue

    ticker, invert = fx_tickers[cur]

    print(f"Downloading {cur}: {ticker}")

    data = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        progress=False,
        auto_adjust=False
    )

    if data.empty:
        print(f"WARNING: no data downloaded for {cur}")
        continue

    data = data.reset_index()

    price_col = "Adj Close" if "Adj Close" in data.columns else "Close"

    temp = data[["Date", price_col]].copy()
    temp.columns = ["date", "fx_rate"]

    if invert:
        temp["fx_to_usd"] = 1 / temp["fx_rate"]
    else:
        temp["fx_to_usd"] = temp["fx_rate"]

    temp["curcdd"] = cur
    temp["date"] = pd.to_datetime(temp["date"])

    fx_list.append(temp[["date", "curcdd", "fx_to_usd"]])

fx = pd.concat(fx_list, ignore_index=True)

fx = fx.sort_values(["curcdd", "date"]).reset_index(drop=True)

display(fx.head())
display(fx.tail())

print("\nFX observations by currency:")
print(fx["curcdd"].value_counts())

USD: set fx_to_usd = 1


,date,curcdd,fx_to_usd
0,2010-01-01,CHF,0.967212
1,2010-01-04,CHF,0.971723
2,2010-01-05,CHF,0.967399
3,2010-01-06,CHF,0.973141
4,2010-01-07,CHF,0.967492


,date,curcdd,fx_to_usd
41638,2025-12-23,ZAR,0.059893
41639,2025-12-24,ZAR,0.059993
41640,2025-12-26,ZAR,0.060142
41641,2025-12-29,ZAR,0.060027
41642,2025-12-30,ZAR,0.059978



FX observations by currency:
curcdd
USD    4173
DKK    4164
EUR    4164
GBP    4164
NOK    4164
PLN    4164
SEK    4164
CZK    4163
ZAR    4162
CHF    4161
Name: count, dtype: int64


In [24]:
# --------------------------------------------------
# 5. Calculate daily FX returns
# --------------------------------------------------

fx = fx.sort_values(["curcdd", "date"]).copy()

fx["fx_return"] = (
    fx.groupby("curcdd")["fx_to_usd"]
    .pct_change()
)

# USD has no FX movement
fx.loc[fx["curcdd"] == "USD", "fx_return"] = 0.0

# First observation per currency has no previous FX rate
fx["fx_return"] = fx["fx_return"].fillna(0.0)

display(fx.head())

,date,curcdd,fx_to_usd,fx_return
0,2010-01-01,CHF,0.967212,0.000000
1,2010-01-04,CHF,0.971723,0.004664
2,2010-01-05,CHF,0.967399,-0.004450
3,2010-01-06,CHF,0.973141,0.005936
4,2010-01-07,CHF,0.967492,-0.005805


In [25]:
# --------------------------------------------------
# 6. Merge FX data into STOXX + Fama-French panel
# --------------------------------------------------

merged["date"] = pd.to_datetime(merged["date"])
fx["date"] = pd.to_datetime(fx["date"])

merged_fx = merged.merge(
    fx[["date", "curcdd", "fx_to_usd", "fx_return"]],
    on=["date", "curcdd"],
    how="left"
)

print("Merged shape:", merged_fx.shape)
print("Missing FX rates:", merged_fx["fx_to_usd"].isna().sum())
print("Missing FX returns:", merged_fx["fx_return"].isna().sum())

display(
    merged_fx[
        ["date", "gvkey", "curcdd", "ret", "ret_w", "fx_to_usd", "fx_return"]
    ].head()
)

Merged shape: (3763472, 37)
Missing FX rates: 9121
Missing FX returns: 9121


,date,gvkey,curcdd,ret,ret_w,fx_to_usd,fx_return
0,2010-01-05,001166,EUR,0.008077,0.008077,1.436596,-0.004022
1,2010-01-06,001166,EUR,0.013501,0.013501,1.440403,0.002650
2,2010-01-07,001166,EUR,-0.023817,-0.023817,1.431803,-0.005971
3,2010-01-08,001166,EUR,0.035516,0.035516,1.441109,0.006499
4,2010-01-11,001166,EUR,-0.029648,-0.029648,1.451126,0.006951


In [26]:
print(merged.columns.tolist())

['gvkey', 'iid', 'date', 'isin', 'prccd', 'ajexdi', 'trfd', 'cshoc', 'curcdd', 'exchg', 'fic', 'tic', 'sedol', 'cusip', 'excntry', 'tpci', 'secstat', 'dlrsni', 'dldtei', 'conm', 'conml', 'gsector', 'ggroup', 'gind', 'gsubind', 'tri', 'ret', 'ret_w', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF', 'Mom']


In [27]:
print(merged_fx.columns.tolist())   # the frame where the 8,021 came from

['gvkey', 'iid', 'date', 'isin', 'prccd', 'ajexdi', 'trfd', 'cshoc', 'curcdd', 'exchg', 'fic', 'tic', 'sedol', 'cusip', 'excntry', 'tpci', 'secstat', 'dlrsni', 'dldtei', 'conm', 'conml', 'gsector', 'ggroup', 'gind', 'gsubind', 'tri', 'ret', 'ret_w', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF', 'Mom', 'fx_to_usd', 'fx_return']


In [28]:
miss = merged_fx[merged_fx['fx_to_usd'].isna()]
print(miss.groupby('curcdd').size().sort_values(ascending=False).to_string())   # which currencies
print()
print(miss['date'].dt.year.value_counts().sort_index().to_string())             # which years
print()
print(miss['date'].value_counts().head(15).to_string())                         # which exact dates

curcdd
EUR    4394
GBP    2190
CHF     969
SEK     839
DKK     292
NOK     292
PLN     118
CZK      22
ZAR       5

date
2011     455
2012     645
2013     890
2017    1860
2019     920
2025    4351

date
2017-11-16    931
2017-07-11    929
2019-05-22    920
2013-10-08    890
2025-01-01    878
2025-04-18    874
2025-04-21    874
2025-12-31    863
2025-12-25    862
2012-12-04    501
2011-04-15    455
2012-01-27     72
2012-01-31     72


In [29]:
# --------------------------------------------------
# 7. Convert local returns to USD returns
# --------------------------------------------------

merged_fx["ret_usd"] = (
    (1 + merged_fx["ret"]) *
    (1 + merged_fx["fx_return"]) - 1
)

merged_fx["ret_w_usd"] = (
    (1 + merged_fx["ret_w"]) *
    (1 + merged_fx["fx_return"]) - 1
)

# Excess returns in USD
merged_fx["excess_return_usd"] = merged_fx["ret_usd"] - merged_fx["RF"]
merged_fx["excess_return_w_usd"] = merged_fx["ret_w_usd"] - merged_fx["RF"]

display(
    merged_fx[
        [
            "date", "gvkey", "curcdd",
            "ret", "fx_return", "ret_usd",
            "ret_w", "ret_w_usd",
            "RF", "excess_return_w_usd"
        ]
    ].head()
)

,date,gvkey,curcdd,ret,fx_return,ret_usd,ret_w,ret_w_usd,RF,excess_return_w_usd
0,2010-01-05,001166,EUR,0.008077,-0.004022,0.004022,0.008077,0.004022,0.0,0.004022
1,2010-01-06,001166,EUR,0.013501,0.002650,0.016187,0.013501,0.016187,0.0,0.016187
2,2010-01-07,001166,EUR,-0.023817,-0.005971,-0.029646,-0.023817,-0.029646,0.0,-0.029646
3,2010-01-08,001166,EUR,0.035516,0.006499,0.042247,0.035516,0.042247,0.0,0.042247
4,2010-01-11,001166,EUR,-0.029648,0.006951,-0.022903,-0.029648,-0.022903,0.0,-0.022903


In [30]:
n0 = len(merged_fx)
merged_fx = merged_fx.dropna(subset=['excess_return_w_usd']).copy()
print(f"dropped {n0 - len(merged_fx):,} rows ({(n0 - len(merged_fx))/n0:.3%}) | {len(merged_fx):,} remain")

# confirm it's actually clean now
print("remaining NaNs in FX/return cols:")
print(merged_fx[['fx_to_usd','fx_return','ret_usd','ret_w_usd',
                 'excess_return_usd','excess_return_w_usd']].isna().sum())

dropped 9,121 rows (0.242%) | 3,754,351 remain
remaining NaNs in FX/return cols:
fx_to_usd              0
fx_return              0
ret_usd                0
ret_w_usd              0
excess_return_usd      0
excess_return_w_usd    0
dtype: int64


In [31]:
# --------------------------------------------------
# 8. Save files
# --------------------------------------------------

fx.to_csv("fx_to_usd_daily_selected_currencies.csv", index=False)

merged_fx.to_parquet(
    "stoxx_600_returns_ff5_mom_usd_2010_2025.parquet",
    index=False
)

print("Saved FX data and USD-converted merged dataset.")

Saved FX data and USD-converted merged dataset.


In [20]:
merged_fx.to_parquet('stoxx600_panel_ff5s_usd_2013_2025.parquet')

In [32]:


PANEL = "stoxx_600_returns_ff5_mom_usd_2010_2025.parquet"
df = pd.read_parquet(PANEL)
df["date"] = pd.to_datetime(df["date"])

need = ["isin","date","excess_return_usd","Mkt-RF","SMB","HML","RMW","CMA","RF","Mom"]
print("missing columns :", [c for c in need if c not in df.columns])          # want []
print("years           :", df["date"].dt.year.min(), "->", df["date"].dt.year.max(),
      " (MUST start 2010)")
print("rows / firms    :", f"{len(df):,}", "/", df["isin"].nunique())
print("\nNaNs in estimation columns (RollingOLS breaks on these):")
print(df[need[2:]].isna().sum().to_string())                                   # want all 0
print("\nscale check |median| (expect all ~0.005-0.01, same order):")
print(df[["excess_return_usd","Mkt-RF","Mom"]].abs().median().to_string())
print("\nduplicate firm-day rows (must be 0):",
      df.duplicated(subset=["isin","date"]).sum())

missing columns : []
years           : 2010 -> 2025  (MUST start 2010)
rows / firms    : 3,754,351 / 1128

NaNs in estimation columns (RollingOLS breaks on these):
excess_return_usd    0
Mkt-RF               0
SMB                  0
HML                  0
RMW                  0
CMA                  0
RF                   0
Mom                  0

scale check |median| (expect all ~0.005-0.01, same order):
excess_return_usd    0.010028
Mkt-RF                 0.0055
Mom                    0.0035

duplicate firm-day rows (must be 0): 0
